# Onboarding - HCP
Lucas Queirós

# Semana 1 -  DataLoader e Tokenização

## 0. Configuração do ambiente

In [1]:
# Verificando versões das libs
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.11.0
tiktoken version: 0.12.0


In [8]:
# Importando corpus1 (Hamlet)
import requests

url = "https://www.gutenberg.org/cache/epub/1524/pg1524.txt"

response = requests.get(url)
corpus1 = response.text
print(corpus1[:300])

The Project Gutenberg eBook of Hamlet
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License incl


In [ ]:
# importando corpus2 (wikipedia sobre shakespeare)
import requests

url = "https://en.wikipedia.org/api/rest_v1/page/summary/William_Shakespeare"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    data = response.json()
    corpus2 = data["extract"]
    print(corpus2[:300])
else:
    print("Erro:", response.status_code)
    print(response.text[:300])

William Shakespeare was an English playwright, poet and actor. He is widely regarded as the greatest writer in the English language and the world's pre-eminent dramatist. He is often called England's national poet and the "Bard of Avon" or simply "the Bard". His extant works, including collaboration


## 1. Tokenizador rustico (regex)

In [ ]:
import re

class SimpleTokenizer:
    """
    Tokenizer simples que converte texto em tokens com IDs utilizando regex e vice-versa
    """

    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        """
        Converte texto em lista de IDs, substitui tokens desconhecidos por <|unk|>
        """
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text) # qualquer um destes simbolos vira ponto de corte
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        """
        Converte lista de IDs de volta em texto, remove os espaços antes das pontuações.
        """
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [ ]:
# Criando vocabulario de tokens com os dois corpus para o tokenizador simples
full_corpus = corpus1 + " <|endoftext|> " + corpus2

def build_vocab(corpus):
    """
    Constrói o vocabulario para o tokenizador rústico, utiliza todos os itens da regex como separador do split,
    remove espaços vazios, cria um set() e remove duplicatas e por fim adiciona os tokens de "unknow",
    "end of text" e "padding".
    """

    tokens = re.split(r'([,.:;?_!"()\']|--|\s)', corpus)
    tokens = [item.strip() for item in tokens if item.strip()]
    all_tokens = sorted(set(tokens))
    all_tokens.extend(["<|unk|>", "<|endoftext|>", "<|pad|>"])
    return {token: i for i, token in enumerate(all_tokens)}

# Passamos todo o corpus pela função e depois pelo tokenizador
vocab = build_vocab(full_corpus)
simple_tokenizer = SimpleTokenizer(vocab)

## 2. DataLoader manual

In [ ]:
# DataLoader manual implementado com numpy
import numpy as np

class SimpleDataLoader:
    """
    Classe que implementa um dataloader e tem a função de organizar todo o pipeline de 
    tratamento do corpus para prosseguir para o treinamento
    """
    def __init__(self, tokens, batch_size, seq_len, shuffle=True, pad_token=0):
        self.tokens = np.array(tokens, dtype=np.int32)
        self.batch_size = batch_size
        self.seq_len = seq_len
        self.shuffle = shuffle
        self.pad_token = pad_token

        chunk_size = seq_len + 1
        n_chunks = len(self.tokens) // chunk_size

        # Verifica se há tokens sobrando que não formam um chunk completo
        remainder = len(self.tokens) % chunk_size
        if remainder > 0:
            # Completa o chunk parcial com padding
            pad_size = chunk_size - remainder
            padding = np.full(pad_size, pad_token, dtype=np.int32)
            self.tokens = np.concatenate([self.tokens, padding])
            n_chunks += 1

        self.chunks = self.tokens.reshape(n_chunks, chunk_size)

    def __iter__(self):
        indices = np.arange(len(self.chunks))

        if self.shuffle:
            np.random.shuffle(indices)

        for start in range(0, len(indices), self.batch_size):
            batch_idx = indices[start : start + self.batch_size]
            batch = self.chunks[batch_idx]

            # Se o último batch tiver menos que batch_size, completa com chunks de padding
            if len(batch) < self.batch_size:
                pad_chunks_needed = self.batch_size - len(batch)
                pad_chunk = np.full((pad_chunks_needed, self.seq_len + 1), self.pad_token, dtype=np.int32)
                batch = np.concatenate([batch, pad_chunk], axis=0)

            x = batch[:, :-1]
            y = batch[:, 1:]

            yield x, y

    def __len__(self):
        return int(np.ceil(len(self.chunks) / self.batch_size))

## 3. Tokenizador GPT-2

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

## 4. Comparação: Shakespeare
métricas: nº tokens, tokens/palavra, tempo de processamento

In [15]:
test1 = corpus1[:100]
test2 = corpus2[:100]

# tiktoken
print("=" * 15 + "TIKTOKEN" + "=" * 15)

print(test1)
integers1 = tokenizer.encode(test1, allowed_special={"<|endoftext|>"})
print(integers1)
tokens1 = tokenizer.decode(integers1)
print(tokens1)

print(test2)
integers2 = tokenizer.encode(test2, allowed_special={"<|endoftext|>"})
print(integers2)
tokens2 = tokenizer.decode(integers2)
print(tokens2)

# tokenizador simples
print("=" * 15 + "SIMPLE TOKENIZER" + "=" * 15)

print(test1)
integers3 = simple_tokenizer.encode(test1)
print(integers3)
tokens3 = simple_tokenizer.decode(integers3)
print(tokens3)

print(test2)
integers4 = simple_tokenizer.encode(test2)
print(integers4)
tokens4 = simple_tokenizer.decode(integers4)
print(tokens4)

===============TIKTOKEN===============
﻿The Project Gutenberg eBook of Hamlet
    
This eBook is for the use of anyone anywhere in the Un
[171, 119, 123, 464, 4935, 20336, 46566, 286, 4345, 1616, 201, 198, 220, 220, 220, 220, 201, 198, 1212, 46566, 318, 329, 262, 779, 286, 2687, 6609, 287, 262, 791]
﻿The Project Gutenberg eBook of Hamlet
    
This eBook is for the use of anyone anywhere in the Un
William Shakespeare was an English playwright, poet and actor. He is widely regarded as the greatest
[17121, 22197, 373, 281, 3594, 711, 29995, 11, 21810, 290, 8674, 13, 679, 318, 6768, 11987, 355, 262, 6000]
William Shakespeare was an English playwright, poet and actor. He is widely regarded as the greatest
===============SIMPLE TOKENIZER===============
﻿The Project Gutenberg eBook of Hamlet
    
This eBook is for the use of anyone anywhere in the Un
[6127, 744, 407, 2396, 3968, 419, 946, 2396, 3330, 2742, 5343, 5666, 3968, 1275, 1277, 3237, 5343, 6128]
﻿The Project Gutenberg eBook of Hamlet 

## 5. Pipeline utilizando pytorch

In [ ]:
# Repetiremos o processo feito acima porem utilizando as ferramentas do pytorch
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

class dataset_gpt2(Dataset):
"""
Classe que herda da classe Dataset do pytorch e serve como base para todo o pipeline de tokenização e 
preparação do corpus
"""
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokeniza todo o texto de uma vez utilizando o tiktoken
        # O parametro allowed_special faz com que o tokenizer não acuse erro ao encontrar token desconhecidos
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Janela deslizante sobre os tokens
        # Cada iteração avança stride posições e recorta uma janela de max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]

            # converte cada janela em tensores e salva na lista de tensores criadas no construtor
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # Metodo obrigatorio da classe, retorna o tamanho 
    def __len__(self):
        return len(self.input_ids)

    # Metodo obrigatorio da classe, o dataloader chama para cada item e agrupa em batches automaticamente
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


# Método que cria todo o dataloader e gerencia a pipeline
def create_dataloader(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # Inicializa o tokenizador carregando o vocabulario usado no GPT2
    tokenizer = tiktoken.get_encoding("gpt2")

    # Inicializa o dataset
    dataset = GPTDataset(txt, tokenizer, max_length, stride)

    # Inicializa o dataloader da classe DataLoader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
        """
        dataset     :   Objeto da classe dataset
        batch_size  :   Tamanho do batch
        shuffle     :   Embaralhar batches
        drop_last   :   Ignora o ultimo batch
        num_workers :   Processos paralelos para carregar dados, 0 = rodar na thread principal
        """

    return dataloader
